# 03 - Performance Analysis

**Financial Portfolio Tracker** | Notebook 3 of 5

---

## Objective

Deep-dive into portfolio performance: cumulative returns, drawdown analysis, rolling return windows, calendar heatmaps, and return attribution by asset class.

## Key Metrics

**Compound Annual Growth Rate (CAGR):**

$$CAGR = \left(\frac{V_f}{V_i}\right)^{1/n} - 1$$

where $V_f$ is the final portfolio value, $V_i$ is the initial value, and $n$ is the number of years. CAGR represents the constant annual growth rate that would produce the same total return over the period -- it smooths out volatility to give a single annualised number.

**Maximum Drawdown (MDD):**

$$MDD = \min_t \left(\frac{V_t}{\max_{s \leq t} V_s} - 1\right)$$

The maximum drawdown captures the worst peak-to-trough decline -- the deepest hole an investor would have experienced. This is arguably the most important risk metric for real investors, because large drawdowns trigger emotional selling.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRADING_DAYS = 252
RISK_FREE_RATE = 0.045

WEIGHTS = {
    "VOO": 0.30, "VXUS": 0.20, "VWO": 0.10,
    "BND": 0.20, "VNQ": 0.10, "GLD": 0.10,
}
TICKERS = list(WEIGHTS.keys())
BENCHMARK = "SPY"

CATEGORY_MAP = {
    "VOO": "US Equity", "VXUS": "International Equity", "VWO": "Emerging Markets",
    "BND": "Fixed Income", "VNQ": "Real Estate", "GLD": "Commodities",
}

In [ ]:
# Load data from previous notebooks
prices = pd.read_parquet(PROCESSED_DIR / "prices_clean.parquet")
returns = pd.read_parquet(PROCESSED_DIR / "returns.parquet")

portfolio_returns = returns["Portfolio"].dropna()
benchmark_returns = returns[BENCHMARK].dropna()

print(f"Data: {len(portfolio_returns)} trading days")
print(f"Period: {portfolio_returns.index.min().date()} to {portfolio_returns.index.max().date()}")

## Cumulative Return Analysis

We visualize the wealth index for each asset alongside the blended portfolio and benchmark.

In [ ]:
# Cumulative returns per asset + portfolio + benchmark
cum_all = pd.DataFrame()
for t in TICKERS:
    cum_all[t] = (1 + returns[t]).cumprod() - 1
cum_all["Portfolio"] = (1 + portfolio_returns).cumprod() - 1
cum_all["SPY"] = (1 + benchmark_returns).cumprod() - 1

fig = go.Figure()

# Individual assets as thin lines
colors = px.colors.qualitative.Set2
for i, t in enumerate(TICKERS):
    fig.add_trace(go.Scatter(
        x=cum_all.index, y=cum_all[t] * 100,
        name=f"{t} ({CATEGORY_MAP[t]})",
        line=dict(width=1, color=colors[i]),
        opacity=0.6,
    ))

# Portfolio as thick blue line
fig.add_trace(go.Scatter(
    x=cum_all.index, y=cum_all["Portfolio"] * 100,
    name="Portfolio", line=dict(width=3, color="#1f77b4"),
))

# Benchmark as thick orange dashed line
fig.add_trace(go.Scatter(
    x=cum_all.index, y=cum_all["SPY"] * 100,
    name="SPY (Benchmark)", line=dict(width=3, color="#ff7f0e", dash="dash"),
))

fig.update_layout(
    title="Cumulative Returns: All Assets, Portfolio & Benchmark",
    yaxis_title="Cumulative Return (%)",
    height=550,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(font=dict(size=10)),
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4)

fig.show()

## Drawdown Analysis

Drawdowns reveal the pain an investor experiences during downturns. We compute the running drawdown from the peak wealth level and identify the worst episodes.

In [ ]:
def compute_drawdown(returns_series):
    """Compute drawdown series from daily returns."""
    wealth = (1 + returns_series).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1
    return drawdown

dd_portfolio = compute_drawdown(portfolio_returns)
dd_benchmark = compute_drawdown(benchmark_returns)

# Drawdown chart with filled area
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dd_portfolio.index, y=dd_portfolio * 100,
    name="Portfolio",
    fill="tozeroy",
    fillcolor="rgba(31, 119, 180, 0.2)",
    line=dict(color="#1f77b4", width=1.5),
))

fig.add_trace(go.Scatter(
    x=dd_benchmark.index, y=dd_benchmark * 100,
    name="SPY (Benchmark)",
    fill="tozeroy",
    fillcolor="rgba(255, 127, 14, 0.15)",
    line=dict(color="#ff7f0e", width=1.5),
))

# Mark max drawdown point
mdd_date = dd_portfolio.idxmin()
mdd_value = dd_portfolio.min()

fig.add_annotation(
    x=mdd_date, y=mdd_value * 100,
    text=f"Max DD: {mdd_value:.1%}",
    showarrow=True, arrowhead=2, arrowsize=1.5,
    font=dict(size=12, color="#d62728"),
)

fig.update_layout(
    title="Drawdown from Peak: Portfolio vs Benchmark",
    yaxis_title="Drawdown (%)",
    height=450,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()

In [ ]:
# Drawdown statistics
def drawdown_stats(returns_series, name):
    wealth = (1 + returns_series).cumprod()
    running_max = wealth.cummax()
    dd = wealth / running_max - 1
    
    trough_date = dd.idxmin()
    max_dd = dd.min()
    peak_date = wealth.loc[:trough_date].idxmax()
    
    # Recovery
    peak_value = wealth.loc[peak_date]
    post_trough = wealth.loc[trough_date:]
    recovered = post_trough[post_trough >= peak_value]
    recovery_date = recovered.index[0] if not recovered.empty else "Not recovered"
    
    # Duration
    dd_days = (trough_date - peak_date).days
    if isinstance(recovery_date, str):
        recovery_days = "N/A"
    else:
        recovery_days = (recovery_date - trough_date).days
    
    return {
        "Name": name,
        "Max Drawdown": f"{max_dd:.2%}",
        "Peak Date": str(peak_date.date()),
        "Trough Date": str(trough_date.date()),
        "Recovery Date": str(recovery_date.date()) if not isinstance(recovery_date, str) else recovery_date,
        "Decline (days)": dd_days,
        "Recovery (days)": recovery_days,
    }

dd_table = pd.DataFrame([
    drawdown_stats(portfolio_returns, "Portfolio"),
    drawdown_stats(benchmark_returns, "SPY (Benchmark)"),
])

print("=" * 70)
print("MAXIMUM DRAWDOWN COMPARISON")
print("=" * 70)
dd_table

## Rolling Return Windows

Rolling returns smooth out daily noise and show how performance varies over different horizons. We use 30-day (1 month), 90-day (1 quarter), and 252-day (1 year) windows.

In [ ]:
# Rolling returns for portfolio
windows = {"30d (1 month)": 30, "90d (1 quarter)": 90, "252d (1 year)": 252}

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=list(windows.keys()),
)

for i, (label, w) in enumerate(windows.items(), 1):
    rolling_p = (1 + portfolio_returns).rolling(w).apply(np.prod, raw=True) - 1
    rolling_b = (1 + benchmark_returns).rolling(w).apply(np.prod, raw=True) - 1
    
    fig.add_trace(
        go.Scatter(x=rolling_p.index, y=rolling_p * 100,
                   name="Portfolio", line=dict(color="#1f77b4", width=1.5),
                   showlegend=(i == 1)),
        row=i, col=1
    )
    fig.add_trace(
        go.Scatter(x=rolling_b.index, y=rolling_b * 100,
                   name="SPY", line=dict(color="#ff7f0e", width=1.5),
                   showlegend=(i == 1)),
        row=i, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4, row=i, col=1)
    fig.update_yaxes(title_text="Return (%)", row=i, col=1)

fig.update_layout(
    title="Rolling Returns: Portfolio vs SPY",
    height=700,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()

## Calendar Return Heatmap

Monthly returns arranged in a year-by-month grid provide an intuitive view of seasonal patterns and help identify which months drove annual performance.

In [ ]:
# Compute monthly returns
monthly_returns = portfolio_returns.groupby(
    [portfolio_returns.index.year, portfolio_returns.index.month]
).apply(lambda x: (1 + x).prod() - 1)

monthly_returns.index.names = ["Year", "Month"]
monthly_table = monthly_returns.unstack(level="Month")
monthly_table.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][:len(monthly_table.columns)]

# Add YTD column
yearly = portfolio_returns.groupby(portfolio_returns.index.year).apply(
    lambda x: (1 + x).prod() - 1
)
monthly_table["YTD"] = yearly.reindex(monthly_table.index).values

print("Monthly Returns (%)")
(monthly_table * 100).round(2)

In [ ]:
# Calendar heatmap
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heatmap_data = monthly_table.drop(columns="YTD", errors="ignore")

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values * 100,
    x=heatmap_data.columns,
    y=heatmap_data.index.astype(str),
    colorscale=[
        [0, "#d32f2f"], [0.25, "#ef9a9a"],
        [0.5, "#ffffff"],
        [0.75, "#a5d6a7"], [1, "#2e7d32"]
    ],
    zmid=0,
    text=np.round(heatmap_data.values * 100, 1),
    texttemplate="%{text:.1f}%",
    textfont={"size": 11},
    colorbar=dict(title="Return (%)"),
))

fig.update_layout(
    title="Portfolio Monthly Returns Heatmap (%)",
    xaxis_title="Month",
    yaxis_title="Year",
    yaxis=dict(autorange="reversed"),
    height=400,
    template="plotly_white",
)

fig.show()

## Return Attribution by Asset Class

How much does each asset contribute to overall portfolio performance? Weighted contribution is:

$$\text{Contribution}_i = w_i \cdot r_i$$

In [ ]:
# Annualised contribution by asset
asset_ann_returns = {}
for t in TICKERS:
    r = returns[t].dropna()
    ann_r = (1 + r).prod() ** (TRADING_DAYS / len(r)) - 1
    asset_ann_returns[t] = ann_r

attribution = pd.DataFrame({
    "Ticker": TICKERS,
    "Category": [CATEGORY_MAP[t] for t in TICKERS],
    "Weight": [WEIGHTS[t] for t in TICKERS],
    "Ann. Return": [asset_ann_returns[t] for t in TICKERS],
    "Contribution": [WEIGHTS[t] * asset_ann_returns[t] for t in TICKERS],
})

attribution["Contribution %"] = attribution["Contribution"] / attribution["Contribution"].sum() * 100

print("=" * 70)
print("RETURN ATTRIBUTION BY ASSET")
print("=" * 70)
attribution

In [ ]:
# Attribution waterfall chart
fig = go.Figure(go.Waterfall(
    name="Contribution",
    orientation="v",
    x=attribution["Ticker"].tolist() + ["Total"],
    y=attribution["Contribution"].tolist() + [None],
    measure=["relative"] * len(TICKERS) + ["total"],
    text=[f"{v:.2%}" for v in attribution["Contribution"]] + [""],
    textposition="outside",
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    increasing={"marker": {"color": "#26a69a"}},
    decreasing={"marker": {"color": "#ef5350"}},
    totals={"marker": {"color": "#1f77b4"}},
))

fig.update_layout(
    title="Return Attribution Waterfall: Contribution by Asset",
    yaxis_title="Annualised Contribution",
    height=450,
    template="plotly_white",
    yaxis_tickformat=".1%",
)

fig.show()

## Performance Summary Table

Consolidate all key performance metrics in a single comparison table.

In [ ]:
def performance_summary(ret, name):
    """Compute key performance metrics for a return series."""
    n_years = len(ret) / TRADING_DAYS
    total_ret = (1 + ret).prod() - 1
    cagr = (1 + total_ret) ** (1 / n_years) - 1
    vol = ret.std() * np.sqrt(TRADING_DAYS)
    sharpe = (cagr - RISK_FREE_RATE) / vol if vol > 0 else 0
    
    # Drawdown
    wealth = (1 + ret).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    
    # Sortino
    excess = ret - RISK_FREE_RATE / TRADING_DAYS
    downside = excess[excess < 0]
    downside_std = np.sqrt((downside ** 2).mean()) * np.sqrt(TRADING_DAYS)
    sortino = (cagr - RISK_FREE_RATE) / downside_std if downside_std > 0 else 0
    
    # Calmar
    calmar = cagr / abs(max_dd) if max_dd != 0 else 0
    
    return {
        "Name": name,
        "Total Return": f"{total_ret:.2%}",
        "CAGR": f"{cagr:.2%}",
        "Volatility": f"{vol:.2%}",
        "Sharpe Ratio": f"{sharpe:.3f}",
        "Sortino Ratio": f"{sortino:.3f}",
        "Max Drawdown": f"{max_dd:.2%}",
        "Calmar Ratio": f"{calmar:.3f}",
        "Best Day": f"{ret.max():.2%}",
        "Worst Day": f"{ret.min():.2%}",
        "% Positive Days": f"{(ret > 0).mean():.1%}",
    }

summary = pd.DataFrame([
    performance_summary(portfolio_returns, "Portfolio"),
    performance_summary(benchmark_returns, "SPY (Benchmark)"),
])

print("=" * 70)
print("PERFORMANCE SUMMARY")
print("=" * 70)
summary.set_index("Name").T

## Summary & Interpretation

**Key Findings:**

1. **Return-risk trade-off**: The diversified portfolio delivers lower absolute returns than SPY (expected, since we hold bonds and international assets) but with meaningfully lower volatility and shallower drawdowns.

2. **Drawdown resilience**: The portfolio's maximum drawdown is notably less severe than SPY's, demonstrating the protective effect of including BND (fixed income) and GLD (gold). During equity sell-offs, these assets tend to hold value or appreciate.

3. **Calendar patterns**: Monthly return heatmaps reveal seasonal patterns. Some months (like March-April) may consistently show different return profiles due to quarterly rebalancing, tax-loss harvesting, and earnings seasons.

4. **Attribution insight**: VOO (at 30% weight) is the dominant return contributor, followed by VNQ and GLD. BND acts primarily as a volatility dampener rather than a return driver.

**Next step**: Notebook 04 -- Risk analytics with VaR, CVaR, beta/alpha decomposition, and volatility regime analysis.